# Journal Finder System Using Data Mining

Student: Sehed  
Course: Data Mining  

## 1. Introduction

This project aims to develop a journal finder system that recommends 
the top 5 journals based on article abstracts. It also applies clustering 
to identify research topics.


## 2. Data Loading

In [20]:
import pandas as pd

df = pd.read_csv(
    r"C:\Users\efnan\OneDrive - Akdeniz Üniversitesi\Belgeler\journal_data.csv",
    header=None
)

df.columns = ["AcademicRecordId", "AbstractText", "JournalName"]

df.head()

,AcademicRecordId,AbstractText,JournalName
0,88652,<p>After using evolutionary techniques for sin...,ACM COMPUTING SURVEYS
1,88653,<p>Distributed data processing is becoming a r...,ACM COMPUTING SURVEYS
2,88654,<p>Logical models of argument formalize common...,ACM COMPUTING SURVEYS
3,88655,<p>In this paper we review studies of the grow...,ACM COMPUTING SURVEYS
4,88656,<p>We survey the current techniques to cope wi...,ACM COMPUTING SURVEYS


## 3. Clean Text

In [21]:
import re

def clean_text(text):
    text = str(text)
    text = re.sub(r'<.*?>', '', text)
    text = text.lower()
    return text

df['clean_text'] = df['AbstractText'].apply(clean_text)

df[['clean_text', 'JournalName']].head()

,clean_text,JournalName
0,after using evolutionary techniques for single...,ACM COMPUTING SURVEYS
1,distributed data processing is becoming a real...,ACM COMPUTING SURVEYS
2,logical models of argument formalize commonsen...,ACM COMPUTING SURVEYS
3,in this paper we review studies of the growth ...,ACM COMPUTING SURVEYS
4,we survey the current techniques to cope with ...,ACM COMPUTING SURVEYS


## 4.Install scikit-learn

In [22]:
!pip install scikit-learn

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.2 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [23]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report

# نحذف المجلات التي لديها أقل من 2 مقالات
journal_counts = df['JournalName'].value_counts()
valid_journals = journal_counts[journal_counts >= 2].index

df_model = df[df['JournalName'].isin(valid_journals)].copy()

X = df_model['clean_text']
y = df_model['JournalName']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

tfidf = TfidfVectorizer(
    stop_words='english',
    max_features=5000,
    ngram_range=(1, 2)
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

model = MultinomialNB()
model.fit(X_train_tfidf, y_train)

y_pred = model.predict(X_test_tfidf)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Accuracy: 0.24853483828955936
                                                                                                                precision    recall  f1-score   support

                                                    2012 18th IEEE International Conference on Networks (ICON)       0.00      0.00      0.00         1
                                                                                         ACM COMPUTING SURVEYS       0.04      0.07      0.05        15
                                                                ACM JOURNAL ON COMPUTING AND CULTURAL HERITAGE       0.00      0.00      0.00         5
                                                     ACM JOURNAL ON EMERGING TECHNOLOGIES IN COMPUTING SYSTEMS       0.00      0.00      0.00        10
                                                                     ACM SIGCOMM COMPUTER COMMUNICATION REVIEW       0.16      0.53      0.24        15
                                                         

C:\Users\efnan\AppData\Roaming\Python\Python312\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\efnan\AppData\Roaming\Python\Python312\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\efnan\AppData\Roaming\Python\Python312\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capital

In [24]:
def recommend_top_5_journals(input_abstract):
    # تنظيف النص
    cleaned = clean_text(input_abstract)
    
    # تحويل إلى أرقام
    vectorized = tfidf.transform([cleaned])
    
    # حساب الاحتمالات
    probabilities = model.predict_proba(vectorized)[0]
    
    # اختيار أفضل 5
    top5_indices = probabilities.argsort()[-5:][::-1]
    
    # عرض النتائج
    results = pd.DataFrame({
        "Rank": range(1, 6),
        "JournalName": model.classes_[top5_indices],
        "Score": probabilities[top5_indices]
    })
    
    return results

In [25]:
test_abstract = """
This paper presents a machine learning approach for data mining 
and classification using text features.
"""

recommend_top_5_journals(test_abstract)

,Rank,JournalName,Score
0,1,JOURNAL OF INTELLIGENT INFORMATION SYSTEMS,0.022022
1,2,DATA MINING AND KNOWLEDGE DISCOVERY,0.014940
2,3,JOURNAL OF MACHINE LEARNING RESEARCH,0.011396
3,4,KNOWLEDGE AND INFORMATION SYSTEMS,0.010639
4,5,INTELLIGENT DATA ANALYSIS,0.009483


In [26]:
from sklearn.cluster import KMeans

# نستخدم نفس TF-IDF
X_tfidf_full = tfidf.fit_transform(df['clean_text'])

# نحدد عدد المجموعات (مثلاً 10)
kmeans = KMeans(n_clusters=10, random_state=42)

kmeans.fit(X_tfidf_full)

# نضيف الكلستر لكل صف
df['Cluster'] = kmeans.labels_

df[['clean_text', 'Cluster']].head()

,clean_text,Cluster
0,after using evolutionary techniques for single...,5
1,distributed data processing is becoming a real...,2
2,logical models of argument formalize commonsen...,4
3,in this paper we review studies of the growth ...,4
4,we survey the current techniques to cope with ...,5


## Clustering

In [27]:
terms = tfidf.get_feature_names_out()
order_centroids = kmeans.cluster_centers_.argsort()[:, ::-1]

for i in range(10):
    print("Cluster", i)
    top_terms = [terms[ind] for ind in order_centroids[i, :10]]
    print(top_terms)
    print()

Cluster 0
['cloud', 'security', 'service', 'services', 'computing', 'scheme', 'cloud computing', 'authentication', 'iot', 'web']

Cluster 1
['network', 'power', 'energy', 'networks', 'channel', 'performance', 'traffic', 'wireless', 'communication', 'proposed']

Cluster 2
['data', 'mining', 'big', 'big data', 'clustering', 'analysis', 'data mining', 'based', 'techniques', 'processing']

Cluster 3
['fuzzy', 'decision', 'linguistic', 'decision making', 'making', 'intuitionistic', 'operator', 'operators', 'intuitionistic fuzzy', 'sets']

Cluster 4
['research', 'software', 'systems', 'information', 'social', 'design', 'user', 'web', 'development', 'based']

Cluster 5
['algorithm', 'optimization', 'problem', 'algorithms', 'problems', 'search', 'proposed', 'based', 'evolutionary', 'solution']

Cluster 6
['sensor', 'nodes', 'energy', 'networks', 'wireless', 'network', 'sensor networks', 'routing', 'wireless sensor', 'wsns']

Cluster 7
['learning', 'classification', 'recognition', 'feature', 'f

## Cluster Interpretation

The topic clusters were interpreted based on the most frequent TF-IDF terms:

- Cluster 0: Cloud Computing and Security
- Cluster 1: Networks and Communications
- Cluster 2: Data Mining and Big Data
- Cluster 3: Fuzzy Systems and Decision Making
- Cluster 4: Software and Information Systems
- Cluster 5: Optimization and Algorithms
- Cluster 6: Wireless Sensor Networks
- Cluster 7: Machine Learning and Classification
- Cluster 8: Image Processing and Computer Vision
- Cluster 9: General Modeling and Analysis

## Conclusion

This project successfully developed a journal recommendation system using article abstracts. 
The system recommends the top 5 journals for a given abstract using TF-IDF and Multinomial Naive Bayes. 
K-Means clustering was also applied to identify major computer science research topics from the dataset.